# Task 4 — Codifica canonica delle regole GER (EvoMine) e confronto con MTM

Input: `lab/task_3/output_data/collegemsg_evomine_1h_s1_raw.txt` (62 regole GER, EvoMine `-s 1 -e 3 -T full -t -d` su granularità 1h).

Obiettivo: applicare la stessa codifica canonica di `encode_motif` (Task 1/MTM) alle regole EvoMine, e confrontarle con la transition matrix MTM del Task 3.

In [1]:
from __future__ import annotations
import os, re
import pandas as pd

OUT_DATA = 'output_data'
os.makedirs(OUT_DATA, exist_ok=True)

RAW_FILE = '../task_3/output_data/collegemsg_evomine_1h_s1_raw.txt'
MTM_FILE = '../task_3/output_data/mtm_transition_matrix.csv'

print('Import OK ✓')


Import OK ✓


## STEP 1 — Parsing del file grezzo EvoMine

Ogni regola nel file grezzo è un blocco:
```
supp=N
<src>(<label>) --<edgelabel>[<x>]--> <dst>(<label>)
...
```

**Nota non ovvia** (verificata in `task_2/EvoMine/file_converters.py`, funzione `obtain_pattern_list`, `mapping_ts = {3: 0, 1: 1}`): nella notazione grezza di EvoMine l'etichetta d'arco **`3` = precondizione (body, t=0)** e **`1` = postcondizione (head, t>0)** — l'opposto di quel che si potrebbe intuire guardando solo i numeri. Usiamo la stessa convenzione qui per restare coerenti col resto del progetto (Task 2).

In [2]:
EDGE_RE = re.compile(r'^(\d+)\(\d+\)\s*--(\d+)\[\d+\]-->\s*(\d+)\(\d+\)$')
SUPP_RE = re.compile(r'^supp=(\d+)$')

BODY_LABEL = 3   # precondizione (t=0)
HEAD_LABEL = 1   # postcondizione (t>0)

def parse_evomine_raw(path):
    """Estrae le regole GER (supp, edges con etichetta 1/3) dal log grezzo di EvoMine."""
    with open(path) as fh:
        lines = fh.readlines()

    start = None
    for i, l in enumerate(lines):
        if l.startswith('saving results to'):
            start = i + 1
            break
    if start is None:
        raise ValueError("Marcatore 'saving results to' non trovato nel file")

    rules = []
    cur = None
    for line in lines[start:]:
        line = line.strip()
        if not line:
            continue
        m = SUPP_RE.match(line)
        if m:
            if cur is not None:
                rules.append(cur)
            cur = {'supp': int(m.group(1)), 'edges': []}
            continue
        m = EDGE_RE.match(line)
        if m:
            src, lbl, dst = int(m.group(1)), int(m.group(2)), int(m.group(3))
            cur['edges'].append((src, dst, lbl))
    if cur is not None:
        rules.append(cur)
    return rules

raw_rules = parse_evomine_raw(RAW_FILE)
print(f'Regole GER parsate: {len(raw_rules)}')

for r in raw_rules[:3]:
    body = [(s, d) for s, d, l in r['edges'] if l == BODY_LABEL]
    head = [(s, d) for s, d, l in r['edges'] if l == HEAD_LABEL]
    print(f"supp={r['supp']:>6}  body={body}  head={head}")


Regole GER parsate: 62
supp=  1941  body=[]  head=[(0, 1)]
supp=   654  body=[]  head=[(0, 1), (0, 2)]
supp=   258  body=[]  head=[(0, 1), (0, 2), (0, 3)]


## STEP 2 — Codifica canonica MTM (`encode_motif`)

Stessa logica di `encode_motif` (Task 1): i nodi vengono rinominati in ordine di prima apparizione (0, 1, 2, ...). Per essere coerenti con la convenzione MTM (prefisso poi evento successivo), gli archi **body vengono processati prima degli archi head** — nell'ordine in cui compaiono nel blocco — anche se nel file grezzo di EvoMine le righe non sono necessariamente stampate in quest'ordine.

In [3]:
def encode_motif(edges):
    """Rinomina i nodi in ordine di prima apparizione e concatena src->dst come stringa."""
    code_map = {}; i = 0; result = []
    for (u, v) in edges:
        if u not in code_map: code_map[u] = str(i); i += 1
        result.append(code_map[u])
        if v not in code_map: code_map[v] = str(i); i += 1
        result.append(code_map[v])
    return ''.join(result)

def encode_rule(rule):
    body_edges = [(s, d) for s, d, l in rule['edges'] if l == BODY_LABEL]
    head_edges = [(s, d) for s, d, l in rule['edges'] if l == HEAD_LABEL]
    combined = body_edges + head_edges
    full_code = encode_motif(combined)
    n_body_digits = 2 * len(body_edges)
    code_body = full_code[:n_body_digits]
    code_head = full_code[n_body_digits:]
    return code_body, code_head

coded_rows = []
for r in raw_rules:
    code_body, code_head = encode_rule(r)
    coded_rows.append({
        'codice_body': code_body,
        'codice_head': code_head,
        'codice_completo': f'{code_body}→{code_head}',
        'supporto': r['supp'],
    })

print(f'Regole codificate: {len(coded_rows)}')
for row in coded_rows[:5]:
    print(row)


Regole codificate: 62
{'codice_body': '', 'codice_head': '01', 'codice_completo': '→01', 'supporto': 1941}
{'codice_body': '', 'codice_head': '0102', 'codice_completo': '→0102', 'supporto': 654}
{'codice_body': '', 'codice_head': '010203', 'codice_completo': '→010203', 'supporto': 258}
{'codice_body': '01', 'codice_head': '0203', 'codice_completo': '01→0203', 'supporto': 563}
{'codice_body': '01', 'codice_head': '1210', 'codice_completo': '01→1210', 'supporto': 134}


## STEP 3 — Tabella risultati


In [4]:
df_coded = pd.DataFrame(coded_rows)
df_coded = df_coded.sort_values('supporto', ascending=False).reset_index(drop=True)
df_coded = df_coded[['codice_body', 'codice_head', 'codice_completo', 'supporto']]

df_coded.to_csv(os.path.join(OUT_DATA, 'evomine_coded_rules.csv'), index=False)
print(f"Salvato: {os.path.join(OUT_DATA, 'evomine_coded_rules.csv')}")
df_coded


Salvato: output_data/evomine_coded_rules.csv


,codice_body,codice_head,codice_completo,supporto
0,01,,01→,32571
1,0112,,0112→,21148
2,0102,,0102→,20136
3,011203,,011203→,19195
4,0110,,0110→,17920
...,...,...,...,...
57,01,2010,01→2010,98
58,01,1020,01→1020,98
59,01,1002,01→1002,55
60,01,1220,01→1220,27


## STEP 4 — Confronto con MTM

La transition matrix MTM (`mtm_transition_matrix.csv`) ha come indice i prefissi a 2 eventi (4 cifre, es. `0102`) e come colonne il terzo evento (2 cifre, es. `03`); le celle sono `P(next_event | prefix)`.

Una regola EvoMine è **strutturalmente comparabile** con la matrice MTM solo se ha esattamente **2 archi body** (prefisso a 4 cifre) e **esattamente 1 arco head** (evento successivo a 2 cifre): la matrice non contiene altre combinazioni di lunghezza. Le regole con body/head di lunghezza diversa (es. body vuoto, head con 2-3 archi) sono automaticamente `in_mtm=False`, indipendentemente da coincidenze testuali nella stringa concatenata.

In [5]:
mtm_raw = pd.read_csv(MTM_FILE, dtype=str)
mtm_raw = mtm_raw.rename(columns={mtm_raw.columns[0]: 'prefix'}).set_index('prefix')

mtm_lookup = {}
for prefix, row in mtm_raw.iterrows():
    for next_event, val in row.items():
        if pd.notna(val) and val != '':
            mtm_lookup[prefix + next_event] = float(val)

print(f'Combinazioni (prefisso, evento successivo) osservate in MTM: {len(mtm_lookup)}')


Combinazioni (prefisso, evento successivo) osservate in MTM: 53


In [6]:
def lookup_mtm(code_body, code_head):
    """Ritorna la probabilità MTM solo se la regola ha la stessa struttura
    (2 archi body + 1 arco head) della transition matrix; altrimenti None."""
    if len(code_body) != 4 or len(code_head) != 2:
        return None
    return mtm_lookup.get(code_body + code_head)

comparison_rows = []
for _, row in df_coded.iterrows():
    prob = lookup_mtm(row['codice_body'], row['codice_head'])
    comparison_rows.append({
        'codice_completo': row['codice_completo'],
        'supporto_evomine': row['supporto'],
        'prob_mtm': prob,
        'in_mtm': prob is not None,
    })

df_comparison = pd.DataFrame(comparison_rows)
df_comparison.to_csv(os.path.join(OUT_DATA, 'mtm_evomine_comparison.csv'), index=False)
print(f"Salvato: {os.path.join(OUT_DATA, 'mtm_evomine_comparison.csv')}")
df_comparison


Salvato: output_data/mtm_evomine_comparison.csv


,codice_completo,supporto_evomine,prob_mtm,in_mtm
0,01→,32571,NaN,False
1,0112→,21148,NaN,False
2,0102→,20136,NaN,False
3,011203→,19195,NaN,False
4,0110→,17920,NaN,False
...,...,...,...,...
57,01→2010,98,NaN,False
58,01→1020,98,NaN,False
59,01→1002,55,NaN,False
60,01→1220,27,NaN,False


## Riepilogo finale

In [7]:
n_in_mtm = int(df_comparison['in_mtm'].sum())
n_only_evomine = int((~df_comparison['in_mtm']).sum())

print(f'Regole EvoMine totali:                  {len(df_comparison)}')
print(f'Regole presenti anche in MTM:            {n_in_mtm}')
print(f'Regole esclusive di EvoMine:             {n_only_evomine}')
print()
print('Top 5 regole EvoMine per supporto:')
top5 = df_comparison.sort_values('supporto_evomine', ascending=False).head(5)
print(top5.to_string(index=False))


Regole EvoMine totali:                  62
Regole presenti anche in MTM:            10
Regole esclusive di EvoMine:             52

Top 5 regole EvoMine per supporto:
codice_completo  supporto_evomine  prob_mtm  in_mtm
            01→             32571       NaN   False
          0112→             21148       NaN   False
          0102→             20136       NaN   False
        011203→             19195       NaN   False
          0110→             17920       NaN   False
